# 数据集类

前面几章我们一直用一个样本（温度 28.1°C，湿度 58%，销量 165）训练模型。
真实的训练需要大量样本反复迭代，才能让模型逐渐收敛。

本章引入**数据集**（Dataset）类，负责管理训练数据：
把数据分为**训练集**和**测试集**，支持按批次读取（**批处理**），
让训练循环可以自动遍历全部数据。

---

## 批处理

**批处理**（Mini-batch）是指每次训练不是用一个样本，而是用一小批样本同时计算梯度，
再统一更新一次参数。批处理有两个好处：

* **效率**：矩阵运算可以并行，一次处理多个样本比逐个处理快得多；
* **稳定性**：多个样本的梯度取平均，比单个样本的梯度噪声更小，训练更稳定。

批处理会改变前向传播中张量的形状：单样本时特征是形如 `(2,)` 的一维向量，
批处理时变成形如 `(batch_size, 2)` 的二维矩阵。
这要求我们对 `Tensor.size` 属性和 `Linear` 层的梯度公式做相应调整。

In [ ]:
from abc import ABC, abstractmethod
import numpy as np

## 基础架构

### 张量

批处理时，特征张量的 `data` 是一个形如 `(batch_size, n_features)` 的二维数组。
此前的 `size` 属性用 `len(self.data)` 返回第一个维度（样本数），
但我们需要的是**每个样本的特征数量**，也就是最后一个维度。

因此把 `size` 改为 `self.data.shape[-1]`：
无论数据是一维（单样本）还是二维（批处理），都能正确返回特征数量。

In [ ]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()
        for parent in self.parents:
            parent.backward()

    @property
    def size(self):
        return self.data.shape[-1]   # 最后一维：每个样本的特征数

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

**基础数据集**（Base Dataset）是一个抽象类，封装了数据加载和迭代读取的逻辑。

**初始化时**，它调用子类实现的 `load()` 方法填充四个数据数组，
然后默认进入训练模式（`self.features` 指向训练集特征）。

**模式切换**：
* `train()`：切换到训练模式，`self.features / self.labels` 指向训练集；
* `eval()`：切换到测试模式，指向测试集。

**数据读取**：
* `__getitem__(index)`：按批次索引读取，返回第 `index` 批的特征和标签张量，
  供训练循环逐批迭代；
* `items()`：一次性返回当前模式下**全部**数据，通常用于评估阶段。

**辅助属性**：
* `shape`：返回 `(输入特征数, 输出特征数)`，用于初始化线性层；
* `__len__`：返回当前模式下的批次总数（样本数 ÷ 批大小）。

``💡 当前数据集按固定顺序读取，没有随机打乱（shuffle）。真实训练中通常需要每轮开始前打乱顺序，以防模型记住样本的排列规律而非学习数据本身。我们会在后续章节引入 shuffle。``

In [ ]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.train_features = None
        self.train_labels   = None
        self.test_features  = None
        self.test_labels    = None

        self.load()                          # 子类填充四个数组

        self.features = self.train_features  # 默认进入训练模式
        self.labels   = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels   = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels   = self.test_labels

    @property
    def shape(self):
        # 返回 (输入特征数, 输出特征数)，用于初始化线性层
        return Tensor(self.features).size, Tensor(self.labels).size

    def items(self):
        # 一次性返回当前模式全部数据（用于评估）
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end   = start + self.batch_size
        return Tensor(self.features[start:end]), Tensor(self.labels[start:end])

### 基础层

In [ ]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [ ]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

优化器沿用上一章的设计，包含 `zero_grad()` 和抽象的 `step()`。

In [ ]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        """将所有参数的梯度清零，在每轮训练开始前调用。"""
        for param in self.parameters:
            param.grad = np.zeros_like(param.data)

    @abstractmethod
    def step(self):
        pass

## 数据

### 数据集（冰激凌销量）

`LRDataset` 加载了我们在第一部分使用的小明冰激凌店数据。
训练集包含 4 条历史记录，测试集包含 1 条用于最终评估。

| 集合 | 温度 (°C) | 湿度 (%) | 销量（支）|
|------|---------|--------|--------|
| 训练 | 22.5 | 72.0 | 95 |
| 训练 | 31.4 | 45.0 | 210 |
| 训练 | 19.8 | 85.0 | 70 |
| 训练 | 27.6 | 63.0 | 155 |
| **测试** | **28.1** | **58.0** | **165** |

In [ ]:
class LRDataset(Dataset):

    def load(self):
        self.train_features = [[22.5, 72.0],
                               [31.4, 45.0],
                               [19.8, 85.0],
                               [27.6, 63.0]]
        self.train_labels   = [[95],
                               [210],
                               [70],
                               [155]]

        self.test_features  = [[28.1, 58.0]]
        self.test_labels    = [[165]]

## 模型

### 线性层

批处理改变了权重梯度的计算方式。

单样本时，输入 `x` 是形如 `(n_features,)` 的一维向量，梯度公式为：

$$
\frac{dL}{dW} = \frac{dL}{dp} \cdot x
$$

批处理时，`x` 变成形如 `(batch, n_features)` 的矩阵，
`p.grad` 变成形如 `(batch, out)` 的矩阵。
为了把批次中所有样本的梯度正确累加，公式变为矩阵乘法：

$$
\frac{dL}{dW} = p_{\text{grad}}^T \cdot x
$$

维度验证：`p_grad.T` 形为 `(out, batch)`，`x` 形为 `(batch, n_features)`，
乘积为 `(out, n_features)`，正好与 `weight.grad` 的形状 `(out, n_features)` 一致。

``💡 这个矩阵形式其实和逐样本累加完全等价，只是用一次矩阵乘法替代了循环，效率更高。``

In [ ]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.ones((out_size, in_size)) / in_size)
        self.bias   = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data    # 批处理矩阵形式
            self.bias.grad   += np.sum(p.grad, axis=0)
            x.grad           += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.data.shape}; bias{self.bias.data.shape}]'

### 均方误差损失函数

批处理时，`y` 和 `p` 都是 `(batch, 1)` 的矩阵。
`np.mean()` 已经在计算损失时对整个批次取了平均，
但梯度 `dL/dp` 还需要除以批大小 `n`，才能与损失的平均语义保持一致：

$$
\frac{dL}{dp_i} = \frac{-2(y_i - p_i)}{n}
$$

这里用 `len(y.data)` 获取批大小（即 `y` 的第一个维度，样本数）。

In [ ]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            n = len(y.data)   # 批大小
            p.grad += -2 * (y.data - p.data) / n

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [ ]:
class SGDOptimizer(Optimizer):

    def step(self):
        for param in self.parameters:
            param.data -= param.grad * self.lr

## 设置

### 学习率

In [ ]:
LEARNING_RATE = 0.00001

### 批大小

批大小（`BATCH_SIZE`）决定每次训练用几个样本。
这里设为 `2`：4 条训练数据分成 2 批，每批 2 个样本。

In [ ]:
BATCH_SIZE = 2

## 训练

### 建模

`dataset.shape` 返回 `(输入特征数, 输出特征数)`，
直接传给 `Linear` 初始化，不需要手动填写 `in_size` 和 `out_size`。

In [ ]:
dataset   = LRDataset(BATCH_SIZE)

# dataset.shape = (输入特征数=2, 输出特征数=1)
layer     = Linear(dataset.shape[0], dataset.shape[1])
print(layer)

loss_fn   = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

### 迭代

现在可以用数据集进行批次迭代训练。
训练循环仍然是上一章建立的四步：
`zero_grad → 前向传播 → backward → step`。

`len(dataset)` 返回批次总数（4 样本 ÷ 批大小 2 = 2 批），
所以这个循环会遍历全部训练数据各一次（1 个 epoch）。

In [ ]:
for i in range(len(dataset)):
    features, labels = dataset[i]

    optimizer.zero_grad()            # 1. 清零梯度
    predictions = layer(features)    # 2. 前向传播
    loss = loss_fn(predictions, labels)
    loss.backward()                  # 3. 自动微分
    optimizer.step()                 # 4. 更新参数

    print(f'batch {i}: loss={loss}')

print(f'weight: {layer.weight}')
print(f'bias:   {layer.bias}')

## 验证

### 推理

训练完成后，切换到测试模式，用测试集评估模型。
`dataset.eval()` 把 `self.features / self.labels` 重新指向测试集；
`dataset.items()` 一次性返回测试集的全部数据（不按批次），直接用于推理。

In [ ]:
dataset.eval()
features, labels = dataset.items()
predictions = layer(features)
print(f'prediction: {predictions}')
print(f'label:      {labels}')

### 评估

In [ ]:
loss = loss_fn(predictions, labels)
print(f'test loss: {loss}')

测试损失约为 `11839`，比初始的 `14871` 有所下降，说明训练是有效的。

不过效果有限——我们只训练了 1 轮（epoch），总共才见过 4 个样本各一次。
测试集（温度 28.1，湿度 58）和训练集的分布也有差异，
这体现了模型**泛化能力**（generalization）的挑战：
在训练集上学到的规律，未必能完美迁移到没见过的数据上。

要让模型真正收敛，还需要多轮迭代。这需要一个完整的模型管理机制——
下一章将引入**模型类**，统一管理多层网络的参数和前向传播。

## 课后练习

1. 把训练改为多轮迭代（例如 100 个 epoch），每轮打印一次训练损失。
   观察损失是否持续下降，以及收敛速度如何。

2. 尝试不同的 `BATCH_SIZE`（如 1、2、4），
   观察批大小对训练稳定性和最终测试损失的影响。

3. 思考：去掉 `optimizer.zero_grad()` 后多轮训练会有什么问题？
   （提示：梯度在每轮会如何变化？）